# Americium-241 Energy Analysis
Load Am-241 waveform data, compute energy integrals, create distributions, and plot sample waveforms.

In [ ]:
# Run settings
max_analyzed_events = -1  # -1 = all events, or set a limit for testing
n_sample_plots = 10  # Number of sample waveforms to plot

# User configuration
from pathlib import Path

WORKDIR = Path("/Users/virgolaema/Software/3det/Osc_Data")
WAVEFORM_DIR = WORKDIR / "Am24_251212"
RESULTS_DIR = Path("./docs/output")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Waveform directory: {WAVEFORM_DIR}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import lecroyparser
except ImportError as exc:
    raise ImportError("lecroyparser required. Install: pip install lecroyparser") from exc

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("am_analysis")

In [ ]:
# Find all waveform files
waveform_files = sorted(WAVEFORM_DIR.glob("*.trc"))
print(f"Found {len(waveform_files)} waveform files")

if max_analyzed_events > 0:
    waveform_files = waveform_files[:max_analyzed_events]
    print(f"Limited to {len(waveform_files)} files for analysis")

if len(waveform_files) == 0:
    raise ValueError(f"No .trc files found in {WAVEFORM_DIR}")

# Show first few files as examples
print(f"\nFirst 5 files:")
for f in waveform_files[:5]:
    print(f"  {f.name}")

In [ ]:
def load_waveform(file_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load a waveform from LeCroy .trc file.
    
    Returns:
        tuple: (time_ns, voltage_v)
    """
    try:
        # Use parseAll=True to get correct voltage data
        scope = lecroyparser.ScopeData(str(file_path), parseAll=True)
        time_s = np.asarray(scope.x, dtype=np.float64)
        time_ns = time_s * 1e9  # Convert to nanoseconds
        
        # Extract voltage data
        if isinstance(scope.y, list) and len(scope.y) > 0:
            voltage_v = np.asarray(scope.y[0], dtype=np.float64)
        else:
            voltage_v = np.asarray(scope.y, dtype=np.float64)
        
        return time_ns, voltage_v
    except Exception as exc:
        logger.error(f"Failed to load {file_path}: {exc}")
        return None, None


def compute_total_energy(time_ns: np.ndarray, voltage_v: np.ndarray) -> float:
    """Compute total energy as integral of baseline-subtracted waveform.
    
    Energy is computed as the integral of |V - baseline| over time.
    
    Returns:
        float: Total energy in V·ns (volt-nanoseconds)
    """
    # Compute baseline from first 10 samples
    baseline = np.mean(voltage_v[:10])
    
    # Subtract baseline
    signal = voltage_v - baseline
    
    # Take absolute value (for negative pulses)
    signal_abs = np.abs(signal)
    
    # Integrate using trapezoidal rule
    try:
        energy = np.trapezoid(signal_abs, time_ns)
    except AttributeError:
        energy = np.trapz(signal_abs, time_ns)
    
    return energy

In [ ]:
# Process all files and compute energies
energies = []
failed_count = 0

print(f"Processing {len(waveform_files)} files...")
for i, wf_file in enumerate(waveform_files):
    if (i + 1) % 500 == 0:
        print(f"  Processed {i + 1}/{len(waveform_files)} files...")
    
    time_ns, voltage_v = load_waveform(wf_file)
    
    if time_ns is None or voltage_v is None:
        failed_count += 1
        continue
    
    energy = compute_total_energy(time_ns, voltage_v)
    energies.append({
        'file': wf_file.name,
        'energy_V_ns': energy
    })

print(f"\nProcessing complete!")
print(f"  Successfully processed: {len(energies)} files")
print(f"  Failed: {failed_count} files")

# Create DataFrame
df = pd.DataFrame(energies)
print(f"\nEnergy statistics:")
print(df['energy_V_ns'].describe())

In [ ]:
# Plot energy distribution
fig, ax = plt.subplots(figsize=(12, 7))

# Create histogram
n, bins, patches = ax.hist(df['energy_V_ns'], bins=100, alpha=0.7, 
                            edgecolor='black', linewidth=0.5, color='green')

ax.set_xlabel('Total Energy (V·ns)', fontsize=12)
ax.set_ylabel('Counts', fontsize=12)
ax.set_title('Am-241 Total Energy Distribution', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, linestyle='--', which='both')

# Add statistics text box
stats_text = f"""Events: {len(df)}
Mean: {df['energy_V_ns'].mean():.3f} V·ns
Std: {df['energy_V_ns'].std():.3f} V·ns
Median: {df['energy_V_ns'].median():.3f} V·ns"""
ax.text(0.98, 0.97, stats_text, transform=ax.transAxes,
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7),
        fontsize=10, family='monospace')

plt.tight_layout()
plt.savefig(RESULTS_DIR / "am241_energy_distribution.png", dpi=150, bbox_inches='tight')
print(f"\nPlot saved to: {RESULTS_DIR / 'am241_energy_distribution.png'}")
plt.show()

In [ ]:
# Plot sample waveforms
n_samples = min(n_sample_plots, len(waveform_files))
fig, axes = plt.subplots(n_samples, 1, figsize=(14, 3*n_samples))
if n_samples == 1:
    axes = [axes]

print(f"\nPlotting {n_samples} sample waveforms...")

# Select evenly spaced files across the dataset
indices = np.linspace(0, len(waveform_files)-1, n_samples, dtype=int)

for i, idx in enumerate(indices):
    wf_file = waveform_files[idx]
    time_ns, voltage_v = load_waveform(wf_file)
    
    if time_ns is None:
        axes[i].text(0.5, 0.5, 'Failed to load', ha='center', va='center')
        axes[i].set_title(f"{wf_file.name} - FAILED")
        continue
    
    baseline = np.mean(voltage_v[:10])
    energy = compute_total_energy(time_ns, voltage_v)
    
    # Plot waveform
    axes[i].plot(time_ns, voltage_v, 'g-', linewidth=0.8, alpha=0.7, label='Signal')
    axes[i].axhline(baseline, color='r', linestyle='--', linewidth=1, label='Baseline')
    
    # Mark peak
    peak_idx = np.argmin(voltage_v) if (voltage_v.min() < baseline) else np.argmax(voltage_v)
    axes[i].plot(time_ns[peak_idx], voltage_v[peak_idx], 'ro', markersize=8, label='Peak')
    
    axes[i].set_xlabel('Time (ns)', fontsize=10)
    axes[i].set_ylabel('Voltage (V)', fontsize=10)
    axes[i].set_title(f"{wf_file.name} | Energy: {energy:.3f} V·ns | Peak: {voltage_v[peak_idx]:.4f} V", 
                      fontsize=10)
    axes[i].grid(True, alpha=0.3)
    axes[i].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "am241_sample_waveforms.png", dpi=150, bbox_inches='tight')
print(f"Sample waveforms saved to: {RESULTS_DIR / 'am241_sample_waveforms.png'}")
plt.show()

In [ ]:
# Plot overlaid waveforms (first 50 for visibility)
fig, ax = plt.subplots(figsize=(12, 7))

n_overlay = min(50, len(waveform_files))
print(f"\nPlotting {n_overlay} overlaid waveforms...")

for i, wf_file in enumerate(waveform_files[:n_overlay]):
    time_ns, voltage_v = load_waveform(wf_file)
    if time_ns is not None:
        ax.plot(time_ns, voltage_v, alpha=0.2, linewidth=0.5, color='green')

ax.set_xlabel('Time (ns)', fontsize=12)
ax.set_ylabel('Voltage (V)', fontsize=12)
ax.set_title(f'Am-241 Overlaid Waveforms (n={n_overlay})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "am241_overlaid_waveforms.png", dpi=150, bbox_inches='tight')
print(f"Overlaid waveforms saved to: {RESULTS_DIR / 'am241_overlaid_waveforms.png'}")
plt.show()

In [ ]:
# Save results to CSV
output_csv = RESULTS_DIR / "am241_energy_data.csv"
df.to_csv(output_csv, index=False)
print(f"\nData saved to: {output_csv}")
print(f"Total events saved: {len(df)}")